# Rwanda House Price Prediction — Complete Jupyter Notebook

**Student:** Nziza Jonathan  
**Programme:** BTech ETT  
**Institution:** RP-Kigali College  
**Academic Year:** 2026

This notebook covers data preparation, exploratory analysis, multiple linear
regression, evaluation, diagnostics, coefficient interpretation, model saving,
and a sample prediction.

## 1. Project Objective

The objective is to build a multiple linear regression model that predicts
`House_Price_Million_RWF` using area, bedrooms, bathrooms, house age,
distance to the city centre, parking spaces, and neighborhood.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_csv("house_price_cleaned.csv")
df.head()

## 2. Dataset Structure

The cleaned dataset contains 91 observations and 9 columns. `House_ID` is
retained for identification but excluded from model predictors.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nSummary statistics:")
display(df.describe(include="all").T)

print("\nMissing values:")
display(df.isna().sum().to_frame("Missing Values"))

print("\nDuplicate rows:", df.duplicated().sum())

## 3. Cleaning Decisions

The supplied cleaning workflow identified 5 duplicate records and 9 missing
cells in the original data. The final cleaned dataset has 91 rows, no duplicate
rows, and no missing cells.

Numerical predictor missing values were treated using median imputation,
`Neighborhood` using mode imputation, and the target using mean imputation.
Potential numeric outliers were assessed with the 1.5×IQR rule and the
sequential filtering workflow produced the final 91-row dataset.

In [ ]:
cleaning_decisions = pd.DataFrame({
    "Problem Found": [
        "Duplicate records",
        "Missing numerical predictors",
        "Missing Neighborhood",
        "Missing target",
        "Potential numeric outliers"
    ],
    "Action Taken": [
        "Removed exact duplicates",
        "Median imputation",
        "Mode imputation",
        "Mean imputation",
        "Sequential 1.5×IQR filtering"
    ],
    "Reason": [
        "Avoid repeated observations influencing the model",
        "Preserve usable observations and reduce sensitivity to extreme values",
        "Use the most common observed category",
        "Retain the observation while avoiding row deletion",
        "Reduce the influence of extreme observations"
    ]
})
display(cleaning_decisions)

## 4. Exploratory Visualisations

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["House_Price_Million_RWF"], bins=12, edgecolor="black")
plt.xlabel("House Price (Million RWF)")
plt.ylabel("Frequency")
plt.title("Distribution of House Prices")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Area_m2"], df["House_Price_Million_RWF"], alpha=0.8)
plt.xlabel("Area (m²)")
plt.ylabel("House Price (Million RWF)")
plt.title("Area vs House Price")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
df.boxplot(column="House_Price_Million_RWF", by="Neighborhood", grid=False)
plt.title("House Price by Neighborhood")
plt.suptitle("")
plt.xlabel("Neighborhood")
plt.ylabel("House Price (Million RWF)")
plt.xticks(rotation=25)
plt.show()

In [ ]:
numeric_cols = [
    "Area_m2", "Bedrooms", "Bathrooms", "House_Age_Years",
    "Distance_to_City_km", "Parking_Spaces",
    "House_Price_Million_RWF"
]

plt.figure(figsize=(9, 6))
plt.imshow(df[numeric_cols].corr(), aspect="auto")
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.colorbar(label="Correlation")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

display(df[numeric_cols].corr().round(3))

## 5. Predictors and Target

All columns except `House_ID` and the target are used as predictors.
`Neighborhood` is categorical and is one-hot encoded with one category
dropped to avoid the dummy-variable trap.

In [ ]:
target = "House_Price_Million_RWF"

numeric_features = [
    "Area_m2", "Bedrooms", "Bathrooms",
    "House_Age_Years", "Distance_to_City_km",
    "Parking_Spaces"
]

categorical_features = ["Neighborhood"]

X = df[numeric_features + categorical_features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 6. Build and Train the Multiple Linear Regression Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    drop="first",
                    handle_unknown="ignore"
                ))
            ]),
            categorical_features
        )
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

print("Model trained successfully!")

## 7. Coefficients

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["regressor"].coef_

coefficient_table = pd.DataFrame({
    "Variable": feature_names,
    "Coefficient": coefficients
})

display(coefficient_table)
print("Intercept:", model.named_steps["regressor"].intercept_)

## 8. Multicollinearity Check

In [ ]:
corr = df[numeric_features].corr()
display(corr.round(3))

print(
    "Bedrooms-Bathrooms correlation:",
    df["Bedrooms"].corr(df["Bathrooms"])
)

## 9. Model Evaluation

In [ ]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

def adjusted_r2(y_true, y_pred, p):
    n = len(y_true)
    r2 = r2_score(y_true, y_pred)
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

p = len(model.named_steps["preprocessor"].get_feature_names_out())

evaluation = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "R²": [
        r2_score(y_train, train_pred),
        r2_score(y_test, test_pred)
    ],
    "Adjusted R²": [
        adjusted_r2(y_train, train_pred, p),
        adjusted_r2(y_test, test_pred, p)
    ],
    "MAE (Million RWF)": [
        mean_absolute_error(y_train, train_pred),
        mean_absolute_error(y_test, test_pred)
    ],
    "RMSE (Million RWF)": [
        mean_squared_error(y_train, train_pred) ** 0.5,
        mean_squared_error(y_test, test_pred) ** 0.5
    ]
})

display(evaluation.round(6))

## 10. Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, test_pred, alpha=0.8)

minimum = min(y_test.min(), test_pred.min())
maximum = max(y_test.max(), test_pred.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual Price (Million RWF)")
plt.ylabel("Predicted Price (Million RWF)")
plt.title("Actual vs Predicted House Prices")
plt.show()

## 11. Residual Diagnostics

In [ ]:
residuals = y_test.values - test_pred

plt.figure(figsize=(8, 5))
plt.scatter(test_pred, residuals, alpha=0.8)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price (Million RWF)")
plt.ylabel("Residual (Million RWF)")
plt.title("Residuals vs Predicted Prices")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=10, edgecolor="black")
plt.xlabel("Residual (Million RWF)")
plt.ylabel("Frequency")
plt.title("Distribution of Test Residuals")
plt.show()

## 12. Coefficient Interpretation

The coefficient signs show the direction of the estimated relationship while
holding the other included predictors constant.

- `Area_m2` has a positive coefficient, so larger floor area is associated
  with a higher predicted price.
- `House_Age_Years` has a negative coefficient, so older houses are
  associated with lower predicted prices in this fitted model.
- `Distance_to_City_km` has a negative coefficient, indicating that greater
  distance is associated with a lower predicted price in the fitted model.

## 13. Save and Reload the Model

In [ ]:
joblib.dump(model, "house_price_model.sav")
print("Model saved successfully!")

loaded_model = joblib.load("house_price_model.sav")

original_predictions = model.predict(X_test)
loaded_predictions = loaded_model.predict(X_test)

print(
    "Same predictions:",
    np.allclose(original_predictions, loaded_predictions)
)

## 14. Predict a New House

In [ ]:
new_house = pd.DataFrame({
    "Area_m2": [180],
    "Bedrooms": [4],
    "Bathrooms": [3],
    "House_Age_Years": [5],
    "Distance_to_City_km": [4],
    "Parking_Spaces": [2],
    "Neighborhood": ["Gasabo"]
})

new_prediction = loaded_model.predict(new_house)[0]

print(
    f"Predicted house price: {new_prediction:.2f} million RWF"
)

## 15. Conclusion and Limitations

The final dataset contains 91 observations after the documented cleaning
process. A multiple linear regression pipeline was trained using an 80/20
train-test split with `random_state=42`.

The test-set metrics provide the main indication of generalisation. The model
should be treated as an estimation tool rather than a guaranteed market value.

Limitations include the relatively small dataset and the use of a linear model,
which may not capture all nonlinear relationships affecting house prices.